# 실습 - 미세조정 (부분 미세조정)

**요리 사진 분류 모델 구축**

- 사전학습 모델은 요리를 분류하지 못하는 상태에서 출발
- 백본은 동결, 분류 헤드만 학습
- 학습 전후 정확도를 수치로 비교

**추론 실습과의 차이**

| 구분 | 추론 | 미세조정 |
|---|---|---|
| 필요 항목 | 이미지 · 모델 | 이미지 · **정답 라벨** · 모델 |
| 평가용 데이터 | 불필요 | 필요 |
| 학습 조건 | 불필요 | 반복 횟수 · 학습률 지정 |
| 소요 시간 | 즉시 | 학습 대기 발생 |

**실습 순서**

- 데이터 준비 — 불러오기 · 구간 분리 · 라벨 확인
- 미세조정 전 확인 — 예측 결과 · 정확도
- 모델 구성 — 백본 + 새 분류 헤드
- 학습 범위 지정 — 백본 동결
- 학습 실행
- 결과 확인 및 저장

---
## 0. 환경 준비

- `datasets` — Hub 데이터셋 내려받기
- `transformers` — 사전학습 모델 · 학습 도구
- `torch` — 연산 수행

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)
import torch

print("준비 완료")
print("GPU 사용 가능:", torch.cuda.is_available())

---
## 1. 데이터 준비

### 1-1. 데이터셋 불러오기

- 데이터셋 — `ethz/food101` (요리 사진 101개 범주)
- 전체 10만 장 중 5,000장만 사용 — 학습 시간 단축
- `train_test_split` — 학습용 · 평가용 분리
- `seed=42` — 동일한 분할 재현

In [ ]:
ds = load_dataset("ethz/food101", split="train[:5000]")
ds = ds.train_test_split(test_size=0.2, seed=42)

ds

### 1-2. 라벨 확인

- `features["label"].names` — 범주 이름 목록
- 정답은 **번호**로 저장 — 이름 목록과 대조 필요
- 이후 예측 결과 해석의 기준

In [ ]:
labels = ds["train"].features["label"].names

print("범주 수:", len(labels))
print("앞 5개:", labels[:5])
print("사용된 범주 수:", len(set(ds["train"]["label"])))

### 1-3. 이미지와 정답 대조

- `ds["test"][0]` — 평가용 첫 번째 항목
- 이 이미지를 **학습 전후 비교**에 사용

In [ ]:
sample = ds["test"][0]

print("정답:", labels[sample["label"]])
sample["image"]

---
## 2. 미세조정 전 확인 (정성)

### 2-1. 사전학습 분류 모델의 예측

- 모델 — `google/vit-base-patch16-224` (ImageNet 1000종 학습)
- 요리 이름이 아닌 라벨이 출력됨
- 원인 — 학습 분류 체계에 **요리 범주 부재**

In [ ]:
clf_pre = pipeline(
    task="image-classification",
    model="google/vit-base-patch16-224"
)

for r in clf_pre(sample["image"]):
    print(r["label"], "-", round(r["score"], 4))

print("정답:", labels[sample["label"]])

**확인 사항**

- 접시 · 국자 등 주변 사물로 예측
- 분류 체계 자체가 다름 — 요리를 맞힐 수 없음
- 해결 — 요리용 분류 헤드를 구성하여 학습

---
## 3. 전처리

- 체크포인트 — `google/vit-base-patch16-224-in21k` (분류 헤드 없는 **백본**)
- `AutoImageProcessor` — 모델별 전처리 규칙 자동 적용
- 수행 내용 — 크기 조정 · 정규화 · 형식 변환
- `convert("RGB")` — 흑백 이미지도 3채널로 통일
- `with_transform` — 사용 시점에 변환, 메모리 절약

**참고** — 파운데이션 모델 백본으로 교체하려면 `ckpt` 를 `facebook/dinov2-base` 로 변경한다. 이 경우 5장의 동결 대상도 `model.dinov2` 로 함께 변경한다.

In [ ]:
ckpt = "google/vit-base-patch16-224-in21k"
processor = AutoImageProcessor.from_pretrained(ckpt)

def transform(batch):
    images = [img.convert("RGB") for img in batch["image"]]
    out = processor(images, return_tensors="pt")
    out["labels"] = batch["label"]
    return out

ds = ds.with_transform(transform)

print("전처리 설정 완료")

---
## 4. 모델 구성

- `AutoModelForImageClassification` — **백본 + 분류 헤드** 구조를 자동 구성
- 백본 — 체크포인트의 사전학습 가중치로 채워짐
- 분류 헤드 — 채울 값이 없어 **무작위 초기화**
- `num_labels` — 헤드의 출력 개수 (101)
- `id2label` · `label2id` — 번호와 이름의 대응

실행 시 경고가 출력된다. 헤드가 새로 생성되었다는 안내이며 정상 동작이다.

In [ ]:
id2label = {i: n for i, n in enumerate(labels)}
label2id = {n: i for i, n in enumerate(labels)}

model = AutoModelForImageClassification.from_pretrained(
    ckpt,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

print(model.classifier)

출력의 `in_features` 는 백본 출력 크기, `out_features` 는 요리 범주 수이다.

이 시점의 헤드는 아직 학습되지 않은 상태이다.

---
## 5. 미세조정 전 확인 (정량)

### 5-1. 정확도 측정 함수

- 평가용 1,000장 전체에 대해 예측과 정답 비교
- 학습 전후 **동일한 함수**로 측정 — 동일 기준 비교

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

def accuracy(model, split="test", n=None):
    model.eval().to(device)
    data = ds[split]
    n = n or len(data)
    correct = 0
    with torch.no_grad():
        for i in range(n):
            item = data[i]
            x = item["pixel_values"].unsqueeze(0).to(device)
            pred = model(x).logits.argmax(-1).item()
            if pred == item["labels"]:
                correct += 1
    return correct / n

print("측정 함수 준비 완료")

### 5-2. 학습 전 정확도

- 헤드가 무작위 초기화 상태 — 예측이 무작위 수준
- 101종 중 찍기 확률 약 1%
- 이 값이 미세조정 성과의 **기준선**

In [ ]:
acc_before = accuracy(model)

print("학습 전 정확도:", round(acc_before, 4))
print("무작위 확률:", round(1 / len(labels), 4))

---
## 6. 학습 범위 지정

- `requires_grad = False` — 해당 파라미터를 갱신하지 않음 (**동결**)
- `model.vit` — 백본 부분
- 학습 대상 — **분류 헤드뿐**
- 앞서 다룬 **부분 미세조정** 방식

전체 미세조정을 수행하려면 이 셀을 건너뛴다.

In [ ]:
for p in model.vit.parameters():
    p.requires_grad = False

total = sum(p.numel() for p in model.parameters())
train = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"전체 파라미터: {total:,}")
print(f"학습 대상    : {train:,} ({train/total:.2%})")

학습 대상이 전체의 0.1% 미만이다. 학습 시간과 메모리가 크게 절감된다.

---
## 7. 학습 설정

- `num_train_epochs` — 전체 데이터 반복 횟수
- `per_device_train_batch_size` — 한 번에 처리할 이미지 수
- `learning_rate` — 한 번에 조정하는 정도
- `remove_unused_columns=False` — 이미지 열 유지에 **필수**
- `output_dir` — 결과 저장 폴더

헤드만 학습하므로 학습률을 상대적으로 크게 설정할 수 있다. 전체 미세조정 시에는 낮게 설정한다.

In [ ]:
args = TrainingArguments(
    output_dir="food_model",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=1e-3,
    remove_unused_columns=False,
    logging_steps=20,
    report_to="none",
)

print("학습 설정 완료")

---
## 8. 학습 실행

- `Trainer` — 학습 과정을 자동으로 수행하는 도구
- 추론은 `pipeline`, 학습은 `Trainer`

**진행 표시 읽기**

- `Training Loss` — 예측과 정답의 차이
- 값이 **감소**하면 학습이 진행 중
- 감소가 멈추면 추가 학습 효과 미미

**참고** — 실행 시 `tokenizer` 관련 오류가 발생하면, `Trainer` 에 `processing_class=processor` 인자를 추가한다.

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
)

trainer.train()

---
## 9. 결과 확인

### 9-1. 예측 비교 (정성)

- 학습 전과 **동일한 이미지** 사용
- 요리 이름을 예측하는지 확인

In [ ]:
clf_post = pipeline(
    task="image-classification",
    model=model,
    image_processor=processor,
    device=0 if torch.cuda.is_available() else -1,
)

for r in clf_post(sample["image"]):
    print(r["label"], "-", round(r["score"], 4))

print("정답:", labels[sample["label"]])

### 9-2. 정확도 비교 (정량)

- 평가용 1,000장 전체 측정
- 학습 전과 **동일한 함수** 사용

In [ ]:
acc_after = accuracy(model)

print("학습 전 정확도:", round(acc_before, 4))
print("학습 후 정확도:", round(acc_after, 4))

**확인 사항**

- 정성 · 정량 모두 향상 — 개별 사례와 전체 성능이 일치
- 백본은 그대로, **헤드 학습만으로** 성능 확보
- 학습 대상은 전체 파라미터의 0.1% 미만

---
## 10. 모델 저장과 재사용

### 10-1. 저장

- 모델과 전처리 규칙을 **함께** 저장
- 전처리 규칙을 함께 저장해야 동일한 결과 재현

In [ ]:
trainer.save_model("food_model")
processor.save_pretrained("food_model")

import os
print(os.listdir("food_model"))

### 10-2. 재사용

- 저장한 폴더를 모델 이름 자리에 지정
- 이후에는 **추론과 동일한 방식**으로 사용

In [ ]:
clf_saved = pipeline(
    task="image-classification",
    model="food_model"
)

clf_saved(sample["image"])

---
## 11. 정리

**수행 내용**

- 사전학습 백본에 요리용 분류 헤드를 구성
- 백본을 동결하고 헤드만 학습 (전체의 0.1% 미만)
- 학습 전후 정확도를 동일 기준으로 비교
- 학습한 모델을 저장하여 재사용

**확인한 사항**

- 분류 체계가 다르면 추론만으로는 해결 불가
- 소량의 학습 대상만으로도 성능 확보 가능
- 사전학습 표현은 그대로 유지됨

**다음 단계**

- 백본 내부까지 조정이 필요한 경우 → **LoRA**
- Hub 업로드 — `push_to_hub("계정명/모델명")`

---